In [ ]:
# drive
from google.colab import drive
drive.mount('/content/drive')

# Packages and ngspice setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

!sudo apt-get install ngspice libngspice0

# Create a symbolic link for libngspice.so if it doesn't exist
# This is often needed because PySpice looks for libngspice.so (without the version number)
!if [ ! -f /usr/lib/x86_64-linux-gnu/libngspice.so ]; then \
    sudo ln -s /usr/lib/x86_64-linux-gnu/libngspice.so.0 /usr/lib/x86_64-linux-gnu/libngspice.so; \
    echo "Created symlink /usr/lib/x86_64-linux-gnu/libngspice.so"; \
fi

# Set the environment variable for PySpice to find ngspice
import os
os.environ['PYSPICE_NGSPICE_PATH'] = '/usr/lib/x86_64-linux-gnu/libngspice.so.0'

!pip install PySpice

# Diagnostic: Find the actual path of libngspice.so
# Please share the output of this command after execution.
!dpkg -L ngspice
!dpkg -L libngspice0

# Further Diagnostics: Check file existence and dependencies
!ls -l /usr/lib/x86_64-linux-gnu/libngspice.so
!ldd /usr/lib/x86_64-linux-gnu/libngspice.so

import PySpice.Logging.Logging as Logging
import logging # Import the standard logging module
from PySpice.Spice.Netlist import Circuit
from PySpice.Unit import *
from PySpice.Spice.Parser import SpiceParser
import re
import torch
import scipy.signal as signal
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import concurrent.futures
import multiprocessing
import random
import glob

# Suppress specific PySpice warnings
logger = Logging.setup_logging()
logger.setLevel(logging.ERROR) # Use logging.ERROR instead of Logging.ERROR


# Netlist Processor (SPICE)

In [ ]:
class NetlistProcessor:
    """
    Motor de análisis de Netlists para el Kaspix Omni-Pipeline.
    Encargado de parsear, validar y extraer la configuración paramétrica
    de circuitos SPICE estándar.
    """

    def __init__(self, file_path):
        self.file_path = file_path
        self.content = ""
        self.params_raw = {}      # Diccionario original (texto, ej: '10k')
        self.params_float = {}    # Diccionario numérico (float, ej: 10000.0)
        self.io_config = {        # Metadatos de entrada/salida
            "input_source": None,
            "output_node": None,
            "ground_check": False
        }
        self.is_parsed = False

        # Carga inmediata
        self._load_file()

    def _load_file(self):
        if not os.path.exists(self.file_path):
            raise FileNotFoundError(f"[Kaspix Error] No se encuentra el archivo: {self.file_path}")

        with open(self.file_path, 'r', encoding='utf-8') as f:
            self.content = f.read()

    def _spice_to_float(self, value_str):
        """
        Convierte sufijos de ingeniería SPICE a float de Python.
        Soporta: T, G, Meg, k, m, u, n, p, f
        """
        value_str = value_str.lower().strip('{} ') # Limpiar llaves y espacios

        # Mapa de multiplicadores SPICE
        suffixes = {
            't': 1e12, 'g': 1e9, 'meg': 1e6, 'k': 1e3,
            'mil': 25.4e-6, 'm': 1e-3, 'u': 1e-6,
            'n': 1e-9, 'p': 1e-12, 'f': 1e-15
        }

        # Regex para separar número y sufijo
        # Busca un número (int o float) seguido opcionalmente de letras
        match = re.match(r'^([\d\.\-\+]+)([a-z]*)$', value_str)

        if not match:
            try:
                return float(value_str) # Intento directo
            except ValueError:
                return None # No es un número parseable (ej: una fórmula compleja)

        number, suffix = match.groups()
        multiplier = 1.0

        if suffix:
            # SPICE es 'greedy' con los sufijos, 'meg' es especial
            if suffix.startswith('meg'):
                multiplier = suffixes['meg']
            elif suffix[0] in suffixes:
                multiplier = suffixes[suffix[0]]

        return float(number) * multiplier

    def analyze(self):
        """
        Ejecuta el análisis léxico del netlist.
        """
        lines = self.content.splitlines()

        # Regex Compilados
        re_param = re.compile(r'\.param\s+(\w+)\s*=\s*(\{?[\w\.\-\+]+\}?)', re.IGNORECASE)
        re_source = re.compile(r'^\s*([vV]\w+)\s+(\w+)\s+(\w+)', re.IGNORECASE)
        re_save = re.compile(r'\.(?:save|print)\s+(?:v|tran)\s*\((.+?)\)', re.IGNORECASE)

        for line in lines:
            line = line.strip()
            if not line or line.startswith('*'): continue

            # 1. Detectar Parámetros (.param)
            pm = re_param.search(line)
            if pm:
                name, val_str = pm.groups()
                # Guardar raw
                self.params_raw[name] = val_str
                # Convertir a float
                val_float = self._spice_to_float(val_str)
                if val_float is not None:
                    self.params_float[name] = val_float

            # 2. Detectar Inputs (Heurística: Nombre contiene 'in' o 'src')
            sm = re_source.match(line)
            if sm:
                s_name, n_pos, n_neg = sm.groups()
                # Priorizamos fuentes que parezcan de señal
                if 'in' in s_name.lower() or 'src' in s_name.lower() or 'sig' in s_name.lower():
                    self.io_config["input_source"] = s_name

            # 3. Detectar Ground
            if ' 0 ' in line or line.endswith(' 0'):
                self.io_config["ground_check"] = True

            # 4. Detectar Output (.save)
            svm = re_save.search(line)
            if svm:
                # Extraer contenido dentro de paréntesis
                nodes = svm.group(1).replace(')', '').replace('(', '').split()
                if nodes:
                    self.io_config["output_node"] = nodes[0] # Tomamos el primero como principal

        # Fallback para Output si no hay .save
        if not self.io_config["output_node"]:
             if re.search(r'\b(out|output|salida)\b', self.content, re.IGNORECASE):
                 self.io_config["output_node"] = "out (Inferred)"
             else:
                 self.io_config["output_node"] = "UNKNOWN (Define .save V(node))"

        self.is_parsed = True
        return self.get_summary()

    def get_summary(self):
        if not self.is_parsed: return "No analizado."
        return {
            "file_path": self.file_path,
            "valid_spice": self.io_config["ground_check"],
            "knobs_detected": list(self.params_float.keys()),
            "knobs_values": self.params_float,
            "input_source": self.io_config["input_source"],
            "output_target": self.io_config["output_node"]
        }

# Dataset Generation

In [ ]:
# ============================================================================
# 0. FUNCIONES AUXILIARES - REPRODUCIBILIDAD
# ============================================================================
def set_global_seed(seed=42):
    """
    Establece semillas aleatorias para reproducibilidad total.

    Args:
        seed: Semilla (int)
    """
    import random
    import numpy as np
    import torch

    # Python random
    random.seed(seed)

    # NumPy random
    np.random.seed(seed)

    # PyTorch random
    torch.manual_seed(seed)

    # Si usas CUDA (GPU)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)  # Para multi-GPU

        # Configuración para reproducibilidad CUDA (más lento pero determinista)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    print(f"✅ Kaspix Seed Locked: {seed}")

# ============================================================
# 1. KASPIX SIGNAL FACTORY (Con Dinámica de Amplitud)
# ============================================================
class KaspixSignalFactory:
    def __init__(self, fs, duration_sec):
        self.fs = fs
        self.duration = duration_sec
        self.n_samples = int(fs * duration_sec)
        self.time_axis = np.linspace(0, duration_sec, self.n_samples)

    def _apply_dynamic_gain(self, y):
        """
        Simula la variación de volumen de entrada (Dinámica).
        """
        target_peak = np.exp(np.random.uniform(np.log(0.05), np.log(1.5)))
        max_val = np.max(np.abs(y))
        if max_val > 1e-9:
            return (y / max_val) * target_peak
        return y

    def _apply_input_noise(self, signal_in):
        """
        [NUEVO] Inyecta un 'Piso de Ruido' realista.
        Simula interferencia eléctrica, térmica o de cables.
        """
        # 1. Decidir si esta muestra tendrá ruido (90% de las veces sí)
        if np.random.rand() > 0.9:
            return signal_in

        # 2. Generar Ruido Base (Blanco)
        noise = np.random.randn(len(signal_in))

        # 3. Determinar nivel de ruido (SNR)
        # Un 'Noise Floor' típico en audio va de -60dB (bueno) a -30dB (malo/vintage)
        # Calculamos la amplitud del ruido basada en una referencia fija (no relativa a la señal)
        # Esto simula que el ruido es constante independientemente de si tocas fuerte o suave.
        noise_level_db = np.random.uniform(-70, -30)
        noise_amplitude = 10 ** (noise_level_db / 20)

        # 4. Mezclar
        noisy_signal = signal_in + (noise * noise_amplitude)

        # Opcional: Recortar picos si se pasa de +/- 1.5V (Safety Clip)
        return np.clip(noisy_signal, -1.5, 1.5)

    # --- Generadores Primitivos ---
    def _pink_noise(self):
        uneven = self.n_samples % 2
        X = np.random.randn(self.n_samples // 2 + 1 + uneven) + 1j * np.random.randn(self.n_samples // 2 + 1 + uneven)
        S = np.sqrt(np.arange(len(X)) + 1.)
        y = (np.fft.irfft(X / S)).real
        if uneven: y = y[:-1]
        # ORDEN FÍSICO: Señal -> Ganancia -> Ruido de Cable
        y = self._apply_dynamic_gain(y)
        return self._apply_input_noise(y)

    def _chirp_log(self):
        f_start = 20
        f_end = np.random.uniform(self.fs/4, self.fs/2 * 0.9)
        y = signal.chirp(self.time_axis, f0=f_start, f1=f_end, t1=self.duration, method='logarithmic')
        y = self._apply_dynamic_gain(y)
        return self._apply_input_noise(y)

    def _step_sequence(self):
        num_steps = np.random.randint(3, 12)
        y = np.zeros_like(self.time_axis)
        indices = np.sort(np.random.choice(np.arange(self.n_samples), num_steps, replace=False))
        levels = np.random.uniform(-0.8, 0.8, num_steps)
        current_idx = 0
        for i, idx in enumerate(indices):
            y[current_idx:idx] = levels[i-1] if i > 0 else 0
            current_idx = idx
        y[current_idx:] = levels[-1]
        # Los steps ya tienen amplitud propia, solo añadimos ruido
        return self._apply_input_noise(y)

    def _multitone(self):
        num_tones = np.random.randint(3, 15)
        y = np.zeros_like(self.time_axis)
        for _ in range(num_tones):
            freq = np.random.uniform(20, self.fs/3)
            phase = np.random.uniform(0, 2*np.pi)
            y += np.sin(2 * np.pi * freq * self.time_axis + phase)
        y = self._apply_dynamic_gain(y)
        return self._apply_input_noise(y)

    def _impulse_train(self):
        y = np.zeros_like(self.time_axis)
        num_impulses = np.random.randint(1, 5)
        indices = np.random.randint(0, self.n_samples, num_impulses)
        y[indices] = 1.0
        # Impulsos limpios suelen ser mejores para analisis, pero ruido leve no daña
        return self._apply_input_noise(y)

    def get_signal(self, recipe):
        keys = list(recipe.keys())
        probs = np.array(list(recipe.values()))
        probs /= probs.sum()
        choice = np.random.choice(keys, p=probs)

        sig = None
        name = "Unknown"

        if choice == 'chirp':
            sig, name = self._chirp_log(), "Chirp Log"
        elif choice == 'pink_noise':
            sig, name = self._pink_noise(), "Pink Noise"
        elif choice == 'step_sequence':
            sig, name = self._step_sequence(), "Step Seq"
        elif choice == 'multitone':
            sig, name = self._multitone(), "Multitone"
        elif choice == 'impulse':
            sig, name = self._impulse_train(), "Impulse"
        elif choice == 'sine':
            f = np.random.uniform(20, 1000)
            raw = np.sin(2*np.pi*f*self.time_axis)
            sig, name = self._apply_input_noise(self._apply_dynamic_gain(raw)), f"Sine {int(f)}Hz"
        elif choice == 'silence_decay':
             raw = np.zeros_like(self.time_axis)
             raw[:int(self.n_samples*0.1)] = np.random.randn(int(self.n_samples*0.1))
             # El silencio DEBE tener ruido de fondo para ser realista
             sig, name = self._apply_input_noise(self._apply_dynamic_gain(raw)), "Silence Decay"
        else:
            sig, name = self._pink_noise(), "Default (Pink)"

        return sig, name

# ============================================================
# 2. WORKER SIMULATOR (Con soporte para Dummy Knob)
# ============================================================
def _simulation_worker(task_payload):
    try:
        file_path = task_payload['file_path']
        input_source = task_payload['input_source']
        output_target = task_payload['output_target']
        fs = task_payload['fs']
        duration = task_payload['duration']
        input_signal = task_payload['input_signal']
        knob_config = task_payload['knob_config']
        task_id = task_payload['id']

        # 1. Construir el circuito
        parser = SpiceParser(path=file_path)
        circuit = parser.build_circuit()

        # 2. Inyección de Fuente PWL (Señal de Audio)
        actual_source_name = None
        for element in circuit.element_names:
            if element.upper() == input_source.upper():
                actual_source_name = element
                break

        if actual_source_name:
            original_source = circuit[actual_source_name]
            n_pos, n_neg = original_source.nodes[0], original_source.nodes[1]
            circuit._elements.pop(actual_source_name)

            time_axis = np.linspace(0, duration, len(input_signal))
            input_data = list(zip(time_axis.astype(float), input_signal.astype(float)))
            circuit.PieceWiseLinearVoltageSource('Input_UES', n_pos, n_neg, values=input_data)
        else:
            return {"status": "error", "msg": f"Fuente {input_source} no hallada", "id": task_id}

        # 3. APLICAR KNOBS (Inyección Directa en Componentes)
        # Esta lógica busca el componente (R1, C1, etc) y cambia su valor físico
        # Si no lo encuentra, lo intenta como parámetro global
        for name, val in knob_config.items():
            if name == 'dummy_param': continue

            if name in circuit.element_names:
                obj = circuit[name]
                val_f = float(val)
                # Mapeo dinámico de atributos según el tipo de componente
                if hasattr(obj, 'resistance'): obj.resistance = val_f
                elif hasattr(obj, 'capacitance'): obj.capacitance = val_f
                elif hasattr(obj, 'inductance'): obj.inductance = val_f # <-- IMPORTANTE PARA WAH
                elif hasattr(obj, 'dc_value'): obj.dc_value = val_f      # Para fuentes V e I
                elif hasattr(obj, 'voltage_gain'): obj.voltage_gain = val_f # Para OpAmps/E
            else:
                # Fallback a parámetro de sistema (.param)
                circuit.parameter(name, val)

        # 4. SIMULACIÓN (Con límites de seguridad)
        # Usamos un step_time ligeramente mayor si es necesario para ayudar a la convergencia
        simulator = circuit.simulator(temperature=25, nominal_temperature=25)

        # [MOD] Añadimos un bloque try específico para la simulación
        try:
            analysis = simulator.transient(step_time=1.0/fs, end_time=duration)
        except Exception as e:
            return {"status": "error", "msg": f"NGSpice Crash: {str(e)}", "id": task_id}

        # 5. EXTRACCIÓN Y LIMPIEZA
        output_signal = None
        # Normalizar nombres de nodos para la búsqueda
        target_clean = output_target.upper().replace('V(', '').replace(')', '')

        for node_name in analysis.nodes:
            if str(node_name).upper() == target_clean:
                output_signal = np.array(analysis.nodes[node_name])
                break

        if output_signal is None:
            return {"status": "error", "msg": f"Nodo {output_target} no encontrado", "id": task_id}

        # 6. FILTRO DE ESTABILIDAD (Anti-Explosión)
        # Si la señal tiene NaNs o voltajes absurdos (ej. > 100V en un pedal de 9V), descartamos
        if np.isnan(output_signal).any() or np.max(np.abs(output_signal)) > 100:
            return {"status": "error", "msg": "Inestabilidad numérica detectada (Overflow)", "id": task_id}

        # Resamplear si el simulador cambió el paso de tiempo
        if len(output_signal) != len(time_axis):
            output_signal = np.interp(time_axis, np.array(analysis.time), output_signal)

        return {
            "status": "ok",
            "id": task_id,
            "y": output_signal.astype(np.float32),
            "x_meta": {
                "knobs": np.array(list(knob_config.values()), dtype=np.float32),
                "knob_names": list(knob_config.keys())
            }
        }

    except Exception as e:
        return {"status": "error", "msg": f"Worker Fail: {str(e)}", "id": task_payload.get('id', -1)}

# ============================================================
# 3. KASPIX GENERATOR (Con inyección de Dummy Knob)
# ============================================================
class KaspixParallelGenerator:
    def __init__(self, processor_result, fs=48000, duration_sec=0.2, recipe=None, seed=42):
        self.meta = processor_result
        self.fs = fs
        self.duration = duration_sec
        self.seed = seed
        self.factory = KaspixSignalFactory(fs, duration_sec)
        self.time_axis = self.factory.time_axis
        self.recipe = recipe if recipe else {"chirp": 1.0}

        if not self.meta['valid_spice']:
            raise ValueError("[Kaspix] Netlist inválido.")

        self.max_workers = multiprocessing.cpu_count()
        print(f"⚡ Kaspix Parallel Engine V6 (Dynamic Gain + Dummy Fix): {self.max_workers} núcleos | Seed: {self.seed}")

    def _sample_knobs(self, variation_pct=0.8):
        nominal = self.meta['knobs_values']

        #Rangos gain y temp
        physics_ranges = {
            'temp_val': (273.15, 373.25), #K
            'gain_val': (5000, 20000) #Ohm
        }

        # [MEJORA #3] Si no hay knobs detectados, inyectar uno falso
        if not nominal:
            # Retornamos un valor fijo (ej. 1.0) para mantener la estructura FiLM
            return {'dummy_param': 1.0}

        sampled = {}
        for name, val in nominal.items():
            if name.lower() in physics_ranges:
                low, high = physics_ranges[name.lower()]
                sampled[name] = np.random.uniform(low, high)
            else:
                delta = val * variation_pct
                new_val = np.random.uniform(val - delta, val + delta)
                if new_val <= 0: new_val = val * 0.01
                sampled[name] = new_val
        return sampled

    def build_dataset(self, n_samples=100, save_path="kaspix_dataset.pt"):
        set_global_seed(self.seed)
        print(f"--- Preparando {n_samples} tareas deterministas ---")

        tasks = []
        pre_generated_signals = {}

        for i in range(n_samples):
            sig_in, sig_type = self.factory.get_signal(self.recipe)
            knobs = self._sample_knobs()

            pre_generated_signals[i] = {
                "audio_in": sig_in.astype(np.float32),
                "signal_type": sig_type
            }

            task = {
                "id": i,
                "file_path": self.meta['file_path'],
                "input_source": self.meta['input_source'],
                "output_target": self.meta['output_target'],
                "fs": self.fs,
                "duration": self.duration,
                "input_signal": sig_in,
                "knob_config": knobs
            }
            tasks.append(task)

        print(f"🚀 Procesando...")
        results_unsorted = []
        with concurrent.futures.ProcessPoolExecutor(max_workers=self.max_workers) as executor:
            results_unsorted = list(tqdm(executor.map(_simulation_worker, tasks), total=n_samples))

        print("📦 Ordenando y verificando consistencia...")
        results_sorted = sorted(results_unsorted, key=lambda x: x['id'])

        dataset_x = []
        dataset_y = []
        success_count = 0

        for res in results_sorted:
            idx = res['id']
            if res['status'] == 'ok':
                input_meta = pre_generated_signals[idx]
                worker_meta = res['x_meta']
                x_entry = {
                    "audio_in": input_meta['audio_in'],
                    "signal_type": input_meta['signal_type'],
                    "knobs": worker_meta['knobs'],
                    "knob_names": worker_meta['knob_names']
                }
                dataset_x.append(x_entry)
                dataset_y.append(res['y'])
                success_count += 1
            else:
                if success_count == 0:
                    print(f"❌ Error en muestra {idx}: {res['msg']}")

        if success_count == 0:
            print("❌ FALLO TOTAL.")
            return [], []

        print(f"💾 Guardando {success_count}/{n_samples} muestras en {save_path}...")
        torch.save({
            "x": dataset_x,
            "y": dataset_y,
            "fs": self.fs,
            "meta": self.meta,
            "recipe": self.recipe,
            "seed": self.seed
        }, save_path)

        return dataset_x, dataset_y

# Execution

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
print("Archivos detectados en la carpeta:")
!ls "/content/drive/MyDrive/universal_filter_ngspice/datasets"

In [ ]:
# Crear el subcircuito del AD590 basado en el Data Sheet
ad590_model = """
.SUBCKT AD590 1 2
* 1uA por cada Kelvin (V1 representa la temperatura)
G1 1 2 VALUE={1u * V(1)}
R_int 1 2 10MEG
.ENDS AD590
"""

# Crear un modelo simplificado del OP27
op27_model = """
.SUBCKT OP27 1 2 3 4 5
E1 5 0 VALUE={V(1,2) * 100k}
RIN 1 2 10MEG
.ENDS OP27
"""

with open("/content/AD590.sub", "w") as f:
    f.write(ad590_model)

with open("/content/OP27.sub", "w") as f:
    f.write(op27_model)

print(" Archivos .sub creados con éxito en /content/")

In [ ]:
# Sobrescribir con nombres en minúsculas para evitar conflictos de Case Sensitivity
with open("/content/ad590.sub", "w") as f:
    f.write(".SUBCKT ad590 1 2\nG1 1 2 VALUE={1u * V(1)}\nR1 1 2 10MEG\n.ENDS ad590")

with open("/content/op27.sub", "w") as f:
    f.write(".SUBCKT op27 1 2 3 4 5\nE1 5 0 VALUE={V(1,2) * 100k}\nRIN 1 2 10MEG\n.ENDS op27")

print(" Modelos actualizados y sincronizados.")


In [ ]:
# ============================================================
# BLOQUE PRINCIPAL DE EJECUCIÓN (KASPIX OMNI-PIPELINE V4.2)
# ============================================================
if __name__ == "__main__":
    # 1. CONFIGURACIÓN GLOBAL Y RUTAS
    # ------------------------------------------------------------
    CIRCUITS_DIR = "drive/MyDrive/universal_filter_ngspice/circuits/sensor"
    DATA_DIR = "drive/MyDrive/universal_filter_ngspice/datasets"
    OUTPUT_DATASET = "kaspix_sensor_dataset_60k.pt"

    os.makedirs(DATA_DIR, exist_ok=True)

    N_SAMPLES_PER_NETLIST = 15000
    SAMPLE_RATE = 48000
    DURATION = 0.08

    RECIPE_MIX = {
        "chirp": 0.25, "pink_noise": 0.20, "multitone": 0.15,
        "step_sequence": 0.20, "sine": 0.10, "impulse": 0.05,
        "silence_decay": 0.05
    }
    RECIPE_MIX_temp = {
        "pink_noise": 0.15,
        "step_sequence": 0.80, "impulse": 0.05
    }

    netlist_files = sorted(glob.glob(os.path.join(CIRCUITS_DIR, "*.cir")))

    if not netlist_files:
        print(f" ERROR: No se encontraron archivos .cir en '{CIRCUITS_DIR}'")
        exit(1)

    # --- MAPEO GENÉRICO DE IDs ---
    # Creamos un diccionario para que cada archivo tenga un ID único fijo
    circuit_mapping = {os.path.basename(f): i for i, f in enumerate(netlist_files)}

    all_data_x = []
    all_data_y = []

    print(f" INICIANDO KASPIX OMNI-PIPELINE MULTI-NETLIST")
    print(f" Mapeo de Circuitos: {circuit_mapping}")

    # 2. BUCLE DE PROCESAMIENTO
    # ------------------------------------------------------------
    for target_netlist in netlist_files:
        netlist_name = os.path.basename(target_netlist)
        current_id = circuit_mapping[netlist_name]

        print(f"\n--- Procesando ID {current_id}: {netlist_name} ---")

        processor = NetlistProcessor(target_netlist)
        circuit_meta = processor.analyze()

        if not circuit_meta['valid_spice']:
            continue

        generator = KaspixParallelGenerator(
            processor_result=circuit_meta,
            fs=SAMPLE_RATE,
            duration_sec=DURATION,
            recipe=RECIPE_MIX_temp #cambiar según netlist
        )

        dx, dy = generator.build_dataset(
            n_samples=N_SAMPLES_PER_NETLIST,
            save_path=os.path.join(DATA_DIR, f"temp_{netlist_name}_dataset.pt")
        )

        # Inyectamos metadatos genéricos en cada muestra
        for sample in dx:
            sample['netlist_origin'] = netlist_name
            sample['circuit_id'] = current_id  # <--- ID NUMÉRICO PARA LA RED
            sample['knob_names'] = list(circuit_meta['knobs_values'].keys())

        all_data_x.extend(dx)
        all_data_y.extend(dy)

    # 3. GUARDADO CONSOLIDADO
    # ------------------------------------------------------------
    save_path = os.path.join(DATA_DIR, OUTPUT_DATASET)
    print(f"\n[Final] Consolidando {len(all_data_x)} muestras...")

    torch.save({
        'x': all_data_x,
        'y': all_data_y,
        'metadata': {
            'fs': SAMPLE_RATE,
            'recipe': RECIPE_MIX_temp,
            'n_samples_per_netlist': N_SAMPLES_PER_NETLIST,
            'duration': DURATION,
            'netlist_names': netlist_files,
            'circuit_mapping': circuit_mapping,
            'n_samples_total': len(all_data_x)
        }
    }, save_path)

    print(f" Dataset maestro guardado en: {save_path}")

In [ ]:
# ============================================================
# BLOQUE DE RESCATE: GENERACIÓN EXCLUSIVA SENSOR QUÍMICO (ID 3)
# ============================================================
if __name__ == "__main__":
    import os
    import torch

    # 1. CONFIGURACIÓN DE RUTAS ESPECÍFICAS
    # ------------------------------------------------------------
    # Forzamos la ruta al archivo específico que faltaba
    TARGET_CIRCUIT = "drive/MyDrive/universal_filter_ngspice/circuits/sensor/sensor_qui.cir"
    DATA_DIR = "drive/MyDrive/universal_filter_ngspice/datasets"

    # Nombre único para no sobreescribir los otros 3 que ya tienes
    OUTPUT_DATASET = "dataset_sensor_3_quimico.pt"

    os.makedirs(DATA_DIR, exist_ok=True)

    # Parámetros técnicos (Mantenemos los mismos del original)
    N_SAMPLES_PER_NETLIST = 15000
    SAMPLE_RATE = 48000
    DURATION = 0.08

    # ID fijo para el sensor químico según tu estructura
    FIXED_ID = 3

    # Usamos la receta de sensores (RECIPE_MIX_temp) que mencionaste
    RECIPE_SENSORS = {
        "pink_noise": 0.15,
        "step_sequence": 0.80,
        "impulse": 0.05
    }

    if not os.path.exists(TARGET_CIRCUIT):
        print(f" ERROR: No se encontró el archivo químico en '{TARGET_CIRCUIT}'")
        exit(1)

    print(f" INICIANDO RESCATE KASPIX: SENSOR QUÍMICO")
    print(f" Forzando ID: {FIXED_ID} para {os.path.basename(TARGET_CIRCUIT)}")

    # 2. PROCESAMIENTO ÚNICO
    # ------------------------------------------------------------
    netlist_name = os.path.basename(TARGET_CIRCUIT)

    # Inicializamos el procesador de Netlist
    processor = NetlistProcessor(TARGET_CIRCUIT)
    circuit_meta = processor.analyze()

    if not circuit_meta['valid_spice']:
        print(" Error en el análisis SPICE del archivo químico.")
        exit(1)

    # Inicializamos el generador paralelo
    generator = KaspixParallelGenerator(
        processor_result=circuit_meta,
        fs=SAMPLE_RATE,
        duration_sec=DURATION,
        recipe=RECIPE_SENSORS
    )

    # Generamos el dataset
    print(f" Generando {N_SAMPLES_PER_NETLIST} muestras... Esto tardará unos 25-30 min.")
    dx, dy = generator.build_dataset(
        n_samples=N_SAMPLES_PER_NETLIST,
        save_path=os.path.join(DATA_DIR, f"temp_quimico_rescue.pt")
    )

    # 3. INYECCIÓN DE METADATOS Y GUARDADO
    # ------------------------------------------------------------
    for sample in dx:
        sample['netlist_origin'] = netlist_name
        sample['circuit_id'] = FIXED_ID  # <--- Forzamos el 3
        sample['knob_names'] = list(circuit_meta['knobs_values'].keys())

    save_path = os.path.join(DATA_DIR, OUTPUT_DATASET)

    torch.save({
        'x': dx,
        'y': dy,
        'metadata': {
            'fs': SAMPLE_RATE,
            'recipe': RECIPE_SENSORS,
            'n_samples_per_netlist': N_SAMPLES_PER_NETLIST,
            'duration': DURATION,
            'netlist_name': netlist_name,
            'circuit_id': FIXED_ID,
            'n_samples_total': len(dx)
        }
    }, save_path)

    print(f"\n PARCIAL COMPLETADO: Archivo guardado en {save_path}")
    print(f" Siguiente paso: Unir este archivo con los otros 3 sensores.")

In [ ]:
import torch
import os

# ============================================================
# BLOQUE DE CONSOLIDACIÓN (KASPIX DATA-MERGE V1.0)
# ============================================================

def consolidar_dataset_maestro():
    # 1. RUTAS Y ARCHIVOS
    DATA_DIR = "drive/MyDrive/universal_filter_ngspice/datasets"

    # Lista de los archivos generados por separado
    # Asegúrate de que los nombres coincidan con los que tienes en Drive
    archivos_parciales = [
        "temp_sensor_dataset.pt",
        "foto_sensor_dataset.pt",
        "mec_sensor_dataset.pt",
        "qui_sensor_dataset.pt"
    ]

    OUTPUT_MASTER = "kaspix_sensor_dataset_60k_master.pt"

    all_x = []
    all_y = []

    print(f" Iniciando consolidación en: {DATA_DIR}")

    # 2. CARGA Y UNIÓN
    # ------------------------------------------------------------
    for nombre in archivos_parciales:
        path = os.path.join(DATA_DIR, nombre)

        if os.path.exists(path):
            print(f" Cargando {nombre}...")
            # Cargamos con map_location='cpu' por seguridad de RAM
            data_parcial = torch.load(path, map_location='cpu', weights_only=False)

            all_x.extend(data_parcial['x'])
            all_y.extend(data_parcial['y'])
            print(f"   -> {len(data_parcial['x'])} muestras añadidas.")
        else:
            print(f" ERROR: No se encontró el archivo {nombre}")

    if not all_x:
        print(" No hay datos para guardar. Abortando.")
        return

    # 3. RECONSTRUCCIÓN DE METADATOS
    # ------------------------------------------------------------
    # Intentamos sacar el mapping del primer archivo o lo creamos manual
    circuit_mapping = {
        "temp_sensor_dataset.pt": 0,
        "foto_sensor_dataset.pt": 1,
        "mec_sensor_dataset.pt": 2,
        "qui_sensor_dataset.pt": 3
    }

    dataset_maestro = {
        'x': all_x,
        'y': all_y,
        'metadata': {
            'fs': 48000,
            'duration': 0.08,
            'n_samples_per_netlist': 15000,
            'n_samples_total': len(all_x),
            'circuit_mapping': circuit_mapping,
            'status': "Consolidado tras rescate"
        }
    }

    # 4. GUARDADO FINAL
    # ------------------------------------------------------------
    save_path = os.path.join(DATA_DIR, OUTPUT_MASTER)
    print(f"\n Guardando Dataset Maestro ({len(all_x)} muestras)...")
    torch.save(dataset_maestro, save_path)

    print(f" ¡LISTO! Dataset consolidado en: {save_path}")
    print(" Ya puedes proceder al entrenamiento de la TCN.")

if __name__ == "__main__":
    consolidar_dataset_maestro()

In [ ]:
import torch
import os

# ============================================================
# BLOQUE DE CONSOLIDACIÓN SEGURO (KASPIX DATA-MERGE V2.0)
# ============================================================

def consolidar_dataset_maestro():
    # 1. RUTAS Y ARCHIVOS
    DATA_DIR = "drive/MyDrive/universal_filter_ngspice/datasets"

    # Lista de archivos parciales originales (los que están sanos)
    archivos_parciales = [
        "temp_sensor_dataset.pt",
        "foto_sensor_dataset.pt",
        "mec_sensor_dataset.pt",
        "qui_sensor_dataset.pt"
    ]

    # Nombre de salida nuevo para evitar errores de archivo corrupto
    OUTPUT_MASTER = "sensor_dataset_v3.pt"

    all_x = []
    all_y = []

    print(f" Iniciando consolidación segura en: {DATA_DIR}")

    # 2. CARGA, INYECCIÓN DE ID Y UNIÓN
    # ------------------------------------------------------------
    # Usamos enumerate para que 'i' sea el circuit_id (0, 1, 2, 3)
    for i, nombre in enumerate(archivos_parciales):
        path = os.path.join(DATA_DIR, nombre)

        if os.path.exists(path):
            print(f" Procesando ID {i}: {nombre}...")
            # Cargamos el parcial
            data_parcial = torch.load(path, map_location='cpu', weights_only=False)

            samples_x = data_parcial['x']
            samples_y = data_parcial['y']

            # INYECCIÓN MANUAL: Aseguramos que cada muestra tenga su circuit_id
            for s in samples_x:
                s['circuit_id'] = i

            all_x.extend(samples_x)
            all_y.extend(samples_y)
            print(f"    {len(samples_x)} muestras inyectadas con ID {i}.")
        else:
            print(f" ERROR: No se encontró el archivo {nombre}. Revisa los nombres en Drive.")

    if not all_x:
        print(" No hay datos para guardar. Abortando.")
        return

    # 3. RECONSTRUCCIÓN DE METADATOS
    # ------------------------------------------------------------
    circuit_mapping = {
        "Térmico": 0,
        "Foto": 1,
        "Mecánico": 2,
        "Químico": 3
    }

    dataset_maestro = {
        'x': all_x,
        'y': all_y,
        'metadata': {
            'fs': 48000,
            'duration': 0.08,
            'n_samples_per_netlist': 15000,
            'n_samples_total': len(all_x),
            'circuit_mapping': circuit_mapping,
            'status': "Consolidado V2 - IDs Inyectados"
        }
    }

    # 4. GUARDADO FINAL (En archivo nuevo)
    # ------------------------------------------------------------
    save_path = os.path.join(DATA_DIR, OUTPUT_MASTER)
    print(f"\n Guardando Dataset Maestro Final ({len(all_x)} muestras)...")

    try:
        torch.save(dataset_maestro, save_path)
        print(f" ¡ÉXITO! Dataset consolidado y reparado en: {save_path}")
    except Exception as e:
        print(f" Error crítico al guardar: {e}")

if __name__ == "__main__":
    consolidar_dataset_maestro()

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch

def analyze_kaspix_dataset(file_path):
    print(f" Cargando dataset: {file_path}...")
    # Cargamos permitiendo pesos (weights_only=False por los diccionarios de x)
    data = torch.load(file_path, map_location='cpu', weights_only=False)

    x_data = data['x']
    y_data = data['y']
    num_samples = len(x_data)

    print(f" Dataset cargado con {num_samples} muestras.")

    # --- 1. ESTADÍSTICAS DE SEÑAL (Corrección de TypeError) ---
    print("\n--- Estadísticas de Señal ---")

    # Convertimos a numpy explícitamente para evitar conflictos con np.std/np.mean
    def to_np(signal):
        if isinstance(signal, torch.Tensor):
            return signal.detach().cpu().numpy()
        return np.array(signal)

    all_means_x = [np.mean(to_np(s['audio_in'])) for s in x_data]
    all_stds_y = [np.std(to_np(s)) for s in y_data]

    print(f"Promedio DC Offset (Input): {np.mean(all_means_x):.6f}")
    print(f"Promedio Desviación Estándar (Output): {np.mean(all_stds_y):.4f}")

    # --- 2. VISUALIZACIÓN DE MUESTRAS ---
    indices = np.random.choice(num_samples, 3, replace=False)
    fig, axes = plt.subplots(3, 2, figsize=(15, 12))

    for i, idx in enumerate(indices):
        # Aseguramos formato numpy para graficar
        audio_in = to_np(x_data[idx]['audio_in'])
        audio_out = to_np(y_data[idx])
        cid = x_data[idx].get('circuit_id', 'N/A')

        # Dominio del Tiempo
        axes[i, 0].plot(audio_in[:500], label='Input', alpha=0.8)
        axes[i, 0].plot(audio_out[:500], label='Target (Output)', alpha=0.7, linestyle='--')
        axes[i, 0].set_title(f"Muestra {idx} - Circuito ID: {cid}")
        axes[i, 0].legend()
        axes[i, 0].grid(True)

        # Respuesta en Frecuencia (Bode)
        f, p_in = welch(audio_in, fs=44100, nperseg=1024)
        _, p_out = welch(audio_out, fs=44100, nperseg=1024)
        mag = 10 * np.log10(p_out / (p_in + 1e-12))

        axes[i, 1].semilogx(f, mag, color='red')
        axes[i, 1].set_title(f"Bode - Circuito {cid}")
        axes[i, 1].set_ylabel("dB")
        axes[i, 1].set_ylim(-60, 10)
        axes[i, 1].grid(True, which="both")

    plt.tight_layout()
    plt.show()

# --- EJECUCIÓN ---
# Asegúrate de que la ruta sea correcta
analyze_kaspix_dataset("/content/drive/MyDrive/universal_filter_ngspice/datasets/datasetcompletos/dataset_master_60k.pt")

In [ ]:
import torch
import numpy as np

def inspect_sensor_dataset(file_path):
    print(f" Cargando Dataset: {file_path}")
    # Cargamos con weights_only=False para procesar los diccionarios internos
    data = torch.load(file_path, map_location='cpu', weights_only=False)

    # 1. Estructura General
    x = data['x']
    y = data['y']
    meta = data.get('metadata', {})

    print("\n" + "="*40)
    print(" RESUMEN ESTRUCTURAL")
    print("="*40)
    print(f"Total de muestras: {len(x)}")
    print(f"Keys en cada muestra 'x': {list(x[0].keys()) if len(x)>0 else 'N/A'}")

    # 2. Detalles de Señales (Inputs/Outputs)
    if len(x) > 0:
        audio_in_shape = np.array(x[0]['audio_in']).shape
        audio_out_shape = np.array(y[0]).shape
        print(f"Longitud de Audio (Samples): {audio_in_shape[0]}")
        print(f"Frecuencia de Muestreo (fs): {meta.get('fs', 'No definida')}")

    # 3. Análisis de Knobs (Los parámetros de los sensores)
    if len(x) > 0 and 'knobs' in x[0]:
        all_knobs = np.array([item['knobs'] for item in x])
        print("\n" + "="*40)
        print(" INFORMACIÓN DE KNOBS (Parámetros)")
        print("="*40)
        print(f"Cantidad de knobs por muestra: {all_knobs.shape[1]}")

        # Intentar obtener nombres si existen
        knob_names = meta.get('knob_names', [f"Knob_{i}" for i in range(all_knobs.shape[1])])

        for i, name in enumerate(knob_names):
            k_min = np.min(all_knobs[:, i])
            k_max = np.max(all_knobs[:, i])
            k_mean = np.mean(all_knobs[:, i])
            print(f" {i}. [{name}] -> Min: {k_min:.4f} | Max: {k_max:.4f} | Promedio: {k_mean:.4f}")

    # 4. Distribución de Circuitos (Sensores)
    if len(x) > 0 and 'circuit_id' in x[0]:
        ids = [item['circuit_id'] for item in x]
        unique_ids, counts = np.unique(ids, return_counts=True)
        mapping = meta.get('circuit_mapping', {})

        print("\n" + "="*40)
        print(" DISTRIBUCIÓN DE SENSORES")
        print("="*40)
        # Invertir el mapeo para ver nombres
        inv_map = {v: k for k, v in mapping.items()}
        for uid, count in zip(unique_ids, counts):
            name = inv_map.get(uid, "Desconocido")
            print(f" ID {uid} ({name}): {count} muestras")

# --- EJECUCIÓN ---
path = "/content/drive/MyDrive/universal_filter_ngspice/datasets/kaspix_sensor_dataset_60k_master_v2.pt"
inspect_sensor_dataset(path)

In [ ]:
import torch
import numpy as np
import pandas as pd # Usaremos pandas para que la tabla se vea impecable

def analizar_rangos_por_sensor(file_path):
    print(f" Analizando rangos físicos en {file_path}...")
    data = torch.load(file_path, weights_only=False, map_location='cpu')
    x_data = data['x']

    # Nombres que definimos para tu proyecto
    sensor_names = {0: "Térmico", 1: "Foto/Luz", 2: "Mecánico", 3: "Químico"}

    resumen = []

    for cid in range(4):
        # Filtramos los knobs solo para este ID
        knobs_id = np.array([m['knobs'] for m in x_data if m['circuit_id'] == cid])

        if len(knobs_id) > 0:
            for k_idx in range(knobs_id.shape[1]):
                resumen.append({
                    "Sensor": sensor_names.get(cid, f"ID {cid}"),
                    "ID": cid,
                    "Parámetro": f"Knob {k_idx}",
                    "Min": knobs_id[:, k_idx].min(),
                    "Max": knobs_id[:, k_idx].max(),
                    "Promedio": knobs_id[:, k_idx].mean()
                })

    # Crear DataFrame para visualizar
    df = pd.DataFrame(resumen)

    # Formatear números para que no salgan en notación científica molesta
    pd.options.display.float_format = '{:,.4f}'.format

    return df

# Ejecutar
path_v2 = "/content/drive/MyDrive/universal_filter_ngspice/datasets/kaspix_sensor_dataset_60k_master_v2.pt"
tabla_rangos = analizar_rangos_por_sensor(path_v2)
display(tabla_rangos)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch

def check_dataset_quality(file_path):
    print(f" Analizando calidad en: {file_path}")
    checkpoint = torch.load(file_path, map_location='cpu', weights_only=False)
    x_data = checkpoint['x']
    y_data = checkpoint['y']

    # 1. Verificar NaNs (Valores nulos que rompen el entrenamiento)
    has_nan = False
    for i in range(100): # Chequeo rápido de las primeras 100
        if np.isnan(x_data[i]['audio_in']).any() or np.isnan(y_data[i]).any():
            has_nan = True
            break
    print(f" ¿Hay valores NaN?: {'SÍ (Peligro)' if has_nan else 'NO (Limpio)'}")

    # 2. Visualización por Sensor (ID 0 al 3)
    fig, axes = plt.subplots(4, 2, figsize=(16, 18))
    plt.subplots_adjust(hspace=0.4)

    for sensor_id in range(4):
        # Buscar una muestra de este sensor
        idx = next(i for i, s in enumerate(x_data) if s['circuit_id'] == sensor_id)

        audio_in = x_data[idx]['audio_in']
        audio_out = y_data[idx]
        knobs = x_data[idx]['knobs']

        # --- COLUMNA 1: TIEMPO (Alineación) ---
        # Hacemos zoom a 300 muestras para ver la fase
        axes[sensor_id, 0].plot(audio_in[:300], label='Estímulo (In)', color='blue', alpha=0.6)
        axes[sensor_id, 0].plot(audio_out[:300], label='Respuesta (Out)', color='orange', alpha=0.9)
        axes[sensor_id, 0].set_title(f"SENSOR ID {sensor_id} - Zoom Tiempo (Alineación)")
        axes[sensor_id, 0].legend()
        axes[sensor_id, 0].grid(True, alpha=0.3)

        # --- COLUMNA 2: FRECUENCIA (Física del Sensor) ---
        f, p_in = welch(audio_in, fs=44100, nperseg=1024)
        _, p_out = welch(audio_out, fs=44100, nperseg=1024)
        mag = 10 * np.log10(p_out / (p_in + 1e-12))

        axes[sensor_id, 1].semilogx(f, mag, color='green')
        axes[sensor_id, 1].set_title(f"Respuesta Frecuencial - Knobs: {np.round(knobs, 2)}")
        axes[sensor_id, 1].set_ylabel("Ganancia (dB)")
        axes[sensor_id, 1].set_xlabel("Hz")
        axes[sensor_id, 1].set_ylim(-60, 20)
        axes[sensor_id, 1].grid(True, which="both", alpha=0.3)

    plt.suptitle("CONTROL DE CALIDAD: DATASET DE SENSORES (MASTER 60K)", fontsize=16)
    plt.show()

# --- Ejecutar ---
path = "/content/drive/MyDrive/universal_filter_ngspice/datasets/datasetcompletos/dataset_master_60k_normalized.pt"
check_dataset_quality(path)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch

def identificar_sensores(file_path):
    data = torch.load(file_path, map_location='cpu', weights_only=False)
    x_data = data['x']
    y_data = data['y']

    plt.figure(figsize=(12, 8))

    for sensor_id in range(4):
        # Tomar 10 muestras de cada sensor para promediar su firma
        indices = [i for i, s in enumerate(x_data) if s['circuit_id'] == sensor_id][:10]

        mags = []
        for idx in indices:
            f, p_in = welch(x_data[idx]['audio_in'], fs=44100, nperseg=1024)
            _, p_out = welch(y_data[idx], fs=44100, nperseg=1024)
            mags.append(10 * np.log10(p_out / (p_in + 1e-12)))

        avg_mag = np.mean(mags, axis=0)
        plt.semilogx(f, avg_mag, label=f"Sensor ID {sensor_id}", linewidth=2)

    plt.title("Firmas de Frecuencia por ID de Sensor")
    plt.xlabel("Frecuencia [Hz]")
    plt.ylabel("Ganancia [dB]")
    plt.legend()
    plt.grid(True, which="both", alpha=0.3)
    plt.show()

identificar_sensores("/content/drive/MyDrive/universal_filter_ngspice/datasets/datasetcompletos/dataset_master_60k_normalized.pt")

In [ ]:
import torch
import numpy as np

def comparar_niveles(file_path, id_a=0, id_b=3):
    print(f" Cargando dataset para comparar...")
    data = torch.load(file_path, map_location='cpu', weights_only=False)
    x, y = data['x'], data['y']

    def get_rms(target_id):
        # Filtramos las señales de ese ID y las pasamos a numpy para el cálculo
        signals_rms = []
        count = 0
        for i, s in enumerate(x):
            if s['circuit_id'] == target_id:
                # Convertimos a numpy explícitamente
                sig = y[i].detach().cpu().numpy() if isinstance(y[i], torch.Tensor) else y[i]
                signals_rms.append(np.sqrt(np.mean(sig**2)))
                count += 1
                if count >= 500: break # Con 500 muestras es suficiente para el promedio

        return np.mean(signals_rms)

    rms_0 = get_rms(id_a)
    rms_3 = get_rms(id_b)

    # Evitar división por cero
    if rms_0 == 0: rms_0 = 1e-12

    diff_db = 20 * np.log10(rms_3 / rms_0)

    print("\n" + "="*40)
    print(f" RESULTADO DE LA COMPARACIÓN")
    print("="*40)
    print(f"  > Nivel RMS ID {id_a}: {rms_0:.6f}")
    print(f"  > Nivel RMS ID {id_b}: {rms_3:.6f}")
    print(f"  > Diferencia: {diff_db:.2f} dB")
    print("-" * 40)

    if abs(diff_db) < 0.1:
        print(" OPINIÓN: Las señales son idénticas en amplitud.")
        print("Es muy probable que ID 0 e ID 3 sean el mismo sensor o compartan el mismo netlist.")
    else:
        print(f" OPINIÓN: El ID {id_b} es {'más fuerte' if diff_db > 0 else 'más débil'}.")

# Ejecutar
comparar_niveles("/content/drive/MyDrive/universal_filter_ngspice/datasets/datasetcompletos/dataset_master_60k_normalized.pt")

In [ ]:
import torch
import numpy as np
import os

def normalize_existing_dataset(input_path, output_path):
    print(f" Cargando dataset original: {input_path}")
    # Cargamos el archivo completo
    checkpoint = torch.load(input_path, map_location='cpu', weights_only=False)

    x_data = checkpoint['x']

    # 1. Extraer todos los knobs para calcular estadísticas
    all_knobs = np.array([sample['knobs'] for sample in x_data])

    # 2. Calcular Min y Max por cada columna (Knob 0 y Knob 1)
    k_min = all_knobs.min(axis=0)
    k_max = all_knobs.max(axis=0)

    print("\n Estadísticas antes de normalizar:")
    print(f"Knob 0 -> Min: {k_min[0]:.4f}, Max: {k_max[0]:.4f}")
    print(f"Knob 1 -> Min: {k_min[1]:.4f}, Max: {k_max[1]:.4f}")

    # 3. Aplicar Normalización Min-Max (0 a 1)
    # Formula: x_norm = (x - min) / (max - min)
    for sample in x_data:
        # Transformamos los knobs de la muestra
        knobs_raw = np.array(sample['knobs'])
        knobs_norm = (knobs_raw - k_min) / (k_max - k_min + 1e-8) # Evitar div por 0
        sample['knobs'] = knobs_norm.astype(np.float32)

    # 4. Actualizar metadatos (Guardamos los min/max para poder "des-normalizar" después)
    if 'metadata' not in checkpoint:
        checkpoint['metadata'] = {}

    checkpoint['metadata']['knobs_min'] = k_min
    checkpoint['metadata']['knobs_max'] = k_max
    checkpoint['metadata']['normalization'] = "Min-Max [0, 1]"

    # 5. Guardar el nuevo dataset optimizado
    print(f"\n Guardando dataset normalizado en: {output_path}")
    torch.save(checkpoint, output_path)
    print(" ¡Listo! Dataset listo para la red FiLM.")

# --- EJECUCIÓN ---
ruta_entrada = "/content/drive/MyDrive/universal_filter_ngspice/datasets/datasetcompletos/dataset_master_60k.pt"
direc_salida = "/content/drive/MyDrive/universal_filter_ngspice/datasets/datasetcompletos"
ruta_salida = os.path.join(direc_salida, "dataset_master_60k_normalized.pt")

normalize_existing_dataset(ruta_entrada, ruta_salida)

In [ ]:
import torch
import matplotlib.pyplot as plt
import os
import numpy as np

# Configuración de rutas
path_master = "/content/drive/MyDrive/universal_filter_ngspice/datasets/datasetcompletos/dataset_master_60k_normalized.pt"

# Diccionario para mapear IDs a nombres reales (puedes mover el orden si descubres que es distinto)
sensor_names = {
    0: "Mecánico",
    1: "Luz",
    2: "Térmico",
    3: "Químico"
}

# Crear la figura (2 filas, 2 columnas)
fig, axs = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle(' Inspección de Sensores Kaspix - Validación de Identidad y Escalas', fontsize=20, fontweight='bold')
axs = axs.flatten()

if os.path.exists(path_master):
    # Cargar el dataset maestro una sola vez
    print(" Cargando dataset maestro...")
    data = torch.load(path_master, weights_only=False, map_location='cpu')
    x_data = data['x']
    y_data = data['y']

    for i in range(4): # Recorrer los 4 IDs de sensores
        # Buscar la primera muestra que coincida con el ID actual
        try:
            idx = next(j for j, sample in enumerate(x_data) if sample['circuit_id'] == i)

            # Extraer señales (convertir a numpy para evitar errores de matplotlib)
            x_audio = x_data[idx]['audio_in']
            if torch.is_tensor(x_audio): x_audio = x_audio.numpy()

            y_out = y_data[idx]
            if torch.is_tensor(y_out): y_out = y_out.numpy()
            y_out = y_out.flatten()

            # Graficar
            axs[i].plot(x_audio, label='Estímulo (Entrada)', color='blue', alpha=0.2, linestyle='--')
            axs[i].plot(y_out, label='Respuesta (Vout)', color='orange', linewidth=1.5)

            # Formato de cada gráfico
            name = sensor_names.get(i, f"Sensor ID {i}")
            axs[i].set_title(f"ID {i}: {name}", fontsize=15, color='darkred', fontweight='bold')
            axs[i].set_ylabel("Amplitud Normalizada")
            axs[i].set_xlabel("Muestras (Tiempo)")
            axs[i].grid(True, which='both', linestyle='--', alpha=0.4)
            axs[i].legend(loc='upper right')

            # Análisis de rango y knobs
            v_min, v_max = y_out.min(), y_out.max()
            knobs = x_data[idx]['knobs']
            info_text = (f"Rango Out: [{v_min:.2f}, {v_max:.2f}]\n"
                         f"Knob 0 (C): {knobs[0]:.3f}\n"
                         f"Knob 1 (R): {knobs[1]:.3f}")

            axs[i].text(0.02, 0.98, info_text, transform=axs[i].transAxes,
                        fontsize=10, verticalalignment='top', family='monospace',
                        bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

        except StopIteration:
            axs[i].text(0.5, 0.5, f"No hay datos para ID {i}", ha='center', va='center')

else:
    print(f" No se encontró el archivo en: {path_master}")

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

# 1. Cargar el dataset (Asegúrate de que la ruta sea correcta)
data = torch.load("/content/drive/MyDrive/universal_filter_ngspice/datasets/kaspix_temp_sensor_dataset.pt", weights_only=False)

idx = 0
sample_x = data['x'][idx] # Es un diccionario
sample_y = data['y'][idx] # Es un array de numpy

# 2. Extraer metadatos REALES del diccionario
knob_values = sample_x['knobs'] # Array con los valores
knob_names = sample_x['knob_names'] # Lista con los nombres ['temp_val', 'gain_res']

# Buscamos la posición de cada parámetro
try:
    temp_idx = knob_names.index('temp_val')
    gain_idx = knob_names.index('gain_res')
    temp_k = knob_values[temp_idx]
    res_gain = knob_values[gain_idx]
except (ValueError, KeyError):
    temp_k, res_gain = 300.0, 10.0 # Fallback

# 3. Graficar
plt.figure(figsize=(12, 5))
plt.plot(sample_y.flatten(), color='orange', label="V_out (Sensor)")
plt.plot(sample_x['audio_in'], color='blue', alpha=0.3, label="V_in (Audio/Temp)")
plt.title(f"Muestra #{idx} | Temp: {temp_k:.2f}K | Ganancia REAL: {res_gain/1e3:.1f}kΩ")
plt.ylabel("Voltaje (V)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
import torch
import numpy as np
import os

base_path = "/content/drive/MyDrive/universal_filter_ngspice/datasets"
datasets = {
    "Térmico": "temp_sensor.cir_dataset.pt",
    "Foto": "temp_sensor_foto.cir_dataset.pt",
    "Mecánico": "temp_sensor_mec.cir_dataset.pt",
    "Químico": "temp_sensor_qui.cir_dataset.pt"
}

print(f"{'SENSOR':<12} | {'MIN VOLT':<12} | {'MAX VOLT':<12} | {'ESTADO'}")
print("-" * 60)

for name, filename in datasets.items():
    path = os.path.join(base_path, filename)
    if os.path.exists(path):
        data = torch.load(path, weights_only=False)
        y = data['y'][0].flatten() # Primera muestra

        v_min, v_max = y.min().item(), y.max().item()

        # Diagnóstico rápido
        if np.isnan(v_min): status = " ERROR: NaN (Overflow)"
        elif abs(v_max) < 1e-6: status = " MUERTO: Señal muy débil (uV)"
        elif abs(v_max) > 1000: status = " ERROR: Voltaje Imposible (kV)"
        else: status = " SANO: Rango correcto"

        print(f"{name:<12} | {v_min:>12.4f} | {v_max:>12.4f} | {status}")
    else:
        print(f"{name:<12} | {'---':>12} | {'---':>12} |  No encontrado")

# How to load data (FiLM) and Parameter Statistics

In [ ]:
import torch
import numpy as np
import os
from torch.utils.data import Dataset, DataLoader

class KaspixDatasetFiLM(Dataset):
    def __init__(self, pt_file_path):
        print(f" Cargando Dataset FiLM desde: {pt_file_path}")

        loaded = torch.load(pt_file_path, weights_only=False)
        self.data_x = loaded['x']
        self.data_y = loaded['y']
        self.meta = loaded['metadata']

        # --- SOLUCIÓN AL ERROR DE FORMA (PADDING) ---
        # 1. Encontrar el número máximo de knobs en cualquier circuito
        self.max_knobs = max(len(item['knobs']) for item in self.data_x)

        # 2. Crear una matriz de ceros y llenarla con los knobs existentes
        # Usamos NaNs temporalmente para que el min/max no se vea afectado por el relleno de ceros
        all_knobs_padded = np.full((len(self.data_x), self.max_knobs), np.nan)

        for i, item in enumerate(self.data_x):
            k = item['knobs']
            all_knobs_padded[i, :len(k)] = k

        # 3. Calcular estadísticas ignorando los NaNs
        self.k_min = torch.from_numpy(np.nanmin(all_knobs_padded, axis=0)).float()
        self.k_max = torch.from_numpy(np.nanmax(all_knobs_padded, axis=0)).float()

        # Evitar división por cero
        self.k_max[self.k_max == self.k_min] += 1e-6

        # 4. Sustituir NaNs por 0 para el procesamiento final
        self.all_knobs_fixed = np.nan_to_num(all_knobs_padded, nan=0.0)

        print(f" Dataset Cargado: {len(self.data_x)} muestras.")
        print(f"   -> Máximo de Knobs detectado: {self.max_knobs}")
        print(f"   -> Mapeo de circuitos: {self.meta['circuit_mapping']}")

    def __len__(self):
        return len(self.data_x)

    def __getitem__(self, idx):
        # A. Audio Input [1, L]
        audio_raw = self.data_x[idx]['audio_in']
        x_audio = torch.from_numpy(audio_raw).float().unsqueeze(0)

        # B. Knobs con Padding y Normalización [max_knobs]
        knobs_raw = torch.from_numpy(self.all_knobs_fixed[idx]).float()
        x_knobs_norm = (knobs_raw - self.k_min) / (self.k_max - self.k_min)
        # Asegurar que el relleno de ceros siga siendo cero tras la normalización
        x_knobs_norm = torch.nan_to_num(x_knobs_norm, nan=0.0)

        # C. Circuit ID (¡Nuevo!)
        # Lo devolvemos como un long tensor para usarlo en capas de Embedding
        circuit_id = torch.tensor(self.data_x[idx]['circuit_id']).long()

        # D. Target Audio [1, L]
        target_raw = self.data_y[idx]
        y_target = torch.from_numpy(target_raw).float().unsqueeze(0)

        return x_audio, x_knobs_norm, circuit_id, y_target

# ============================================================
# BLOQUE DE PRUEBA ACTUALIZADO (Unpack de 4 valores)
# ============================================================
if __name__ == "__main__":
    # Asegúrate de que la ruta coincida con tu carpeta de datasets
    ds = KaspixDatasetFiLM("datasets/kaspix_full_dataset.pt")
    dl = DataLoader(ds, batch_size=4, shuffle=True)

    # Ahora desempaquetamos 4 valores: Audio, Knobs, ID, Target
    b_audio, b_knobs, b_ids, b_target = next(iter(dl))

    print("\n--- INSPECCIÓN DE TENSOR (Omni-Pipeline V4.2) ---")
    print(f"1. Audio Input Shape:  {b_audio.shape}")   # (Batch, 1, Time)
    print(f"2. Knobs (Padded):     {b_knobs.shape}")   # (Batch, max_knobs)
    print(f"3. Circuit IDs:        {b_ids.tolist()}")  # Lista de IDs en el batch
    print(f"4. Target Shape:       {b_target.shape}")

    # Ejemplo detallado de la primera muestra del batch
    print(f"\nEjemplo Muestra #0:")
    id_actual = b_ids[0].item()
    # Invertimos el mapping para saber el nombre
    nombres = {v: k for k, v in ds.meta['circuit_mapping'].items()}
    print(f"   Tipo de Filtro: {nombres[id_actual]} (ID: {id_actual})")
    print(f"   Knobs Norm:     {b_knobs[0].numpy()}")